# Table I · Hackability of common predictors

**In the paper ([arXiv:2610.10899](https://arxiv.org/abs/2610.10899)):** Table I and the benchmark paragraphs of Sec. V ("Common predictors differ by more than an order of magnitude", the round-trip offsets).

SSL-MOS, UTMOS22, NISQA and DNSMOS, each attacked by five independently trained attackers on 500 BVCC test
utterances: reconstruction offset, mean and worst-case ASR, mean score gain. Recomputes the benchmark (4 jobs,
~15 min each; released attackers, no training).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['bench']]
print(len(KEYS), "runs:", ", ".join(KEYS))

4 runs: bench_sslmos, bench_utmos, bench_nisqa, bench_dnsmos


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    4
Name: count, dtype: int64

In [6]:
print((PAPER / "tables" / "table_bench.tex").read_text())

% generated by analysis/paper_assets.py
\begin{tabular}{l c c c c}
\toprule
Predictor & offset & mean ASR & worst ASR & mean gain \\
\midrule
SSL-MOS (BVCC) & -0.10 & 0.205 & 0.252 & +0.191 \\
UTMOS22 & -0.07 & 0.057 & 0.076 & +0.144 \\
NISQA & +0.14 & 0.896 & 0.916 & +0.954 \\
DNSMOS OVRL & +0.02 & 0.136 & 0.200 & +0.124 \\
\bottomrule
\end{tabular}



In [7]:
print(repro.diff_text(PAPER / "tables" / "table_bench.tex", COMMITTED_PAPER / "tables" / "table_bench.tex"))

identical to the committed file


In [8]:
pd.read_csv(OUT / "bench_summary.csv").set_index("target")[
    ["recon_gain_mean", "asr_ref_input_mean", "asr_ref_recon_mean", "asr_ref_recon_min", "asr_ref_recon_max",
     "gain_ref_recon_mean"]].round(3)

,recon_gain_mean,asr_ref_input_mean,asr_ref_recon_mean,asr_ref_recon_min,asr_ref_recon_max,gain_ref_recon_mean
target,,,,,,
sslmos,-0.100,0.170,0.205,0.136,0.252,0.191
utmos,-0.069,0.094,0.057,0.042,0.076,0.144
nisqa,0.139,0.839,0.896,0.876,0.916,0.954
dnsmos,0.016,0.183,0.136,0.064,0.200,0.124


In [9]:
repro.compare_csv(OUT / "bench_summary.csv", COMMITTED_OUT / "bench_summary.csv")

n_attackers            0.0
n_utts                 0.0
tau                    0.0
eps                    0.0
asr_ref_recon_mean     0.0
asr_ref_recon_std      0.0
asr_ref_recon_max      0.0
asr_ref_recon_min      0.0
asr_ref_input_mean     0.0
asr_ref_input_std      0.0
asr_ref_input_max      0.0
asr_ref_input_min      0.0
gain_ref_recon_mean    0.0
gain_ref_recon_std     0.0
gain_ref_recon_max     0.0
gain_ref_recon_min     0.0
recon_gain_mean        0.0
recon_gain_std         0.0
recon_gain_max         0.0
recon_gain_min         0.0
Name: max_abs_diff, dtype: float64

In [10]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('bench_',))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

96 numbers; max |diff| = 0.0000; identical: 96


,paper,reproduced,abs_diff
bench_dnsmos_asr_ref_input_mean_2dp,0.180,0.180,0.0
bench_dnsmos_asr_ref_input_mean_3dp,0.183,0.183,0.0
bench_dnsmos_asr_ref_input_mean_pct,18.300,18.300,0.0
bench_dnsmos_asr_ref_input_mean_pct0,18.000,18.000,0.0
bench_dnsmos_asr_ref_recon_max_2dp,0.200,0.200,0.0
bench_dnsmos_asr_ref_recon_max_3dp,0.200,0.200,0.0
bench_dnsmos_asr_ref_recon_max_pct,20.000,20.000,0.0
bench_dnsmos_asr_ref_recon_max_pct0,20.000,20.000,0.0
bench_dnsmos_asr_ref_recon_mean_2dp,0.140,0.140,0.0
bench_dnsmos_asr_ref_recon_mean_3dp,0.136,0.136,0.0
